# FREM-S12 Kaggle campaign launcher
This notebook delegates to the hash-bound package. Qualification and generated smoke do not consume market outcomes. Scientific cells remain receipt-gated.

In [ ]:
from pathlib import Path

MODE = "qualify-kaggle"  # qualify-kaggle | generated-smoke | run-search-cell | select-search | run-cell | analyze
EXECUTE = False
INPUT_ROOT = Path("/kaggle/input")
OUTPUT = Path("/kaggle/working/frem-s12-output")
SITE = Path("/kaggle/working/frem-s12-site")
PACKAGE_VERSION = "0.1.5"
WHEEL_NAME = "frem_s12-0.1.5-py3-none-any.whl"
WHEEL_SHA256 = "052caa05cfb4f5eec70a2744bf5429c7a1c8d12db48ec3d8aa69d7082c9373ba"
FAMILY = "size_bm"
FOLD = "A"
TRIAL = 0
SEED = 11
TOTAL_SHARDS = 8
SHARD = 0

In [ ]:
import hashlib
import json
import os
import platform
import subprocess
import sys

import numpy as np

SUPPORTED_RUNTIMES = {(3, 12): "2.5.4", (3, 13): "2.1.3"}
if OUTPUT.exists() or OUTPUT.is_symlink():
    raise RuntimeError(f"Output root must be absent for a fresh run: {OUTPUT}")
if SITE.exists() or SITE.is_symlink():
    raise RuntimeError(f"Isolated install target must be absent: {SITE}")
OUTPUT.mkdir(parents=True, exist_ok=False)

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_new_json(path: Path, value: object) -> None:
    with path.open("x", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, sort_keys=True, allow_nan=False)
        handle.write("\n")

def one_input_file(name: str) -> Path:
    matches = [path for path in INPUT_ROOT.rglob(name) if path.is_file() and not path.is_symlink()]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one input file named {name}; found {len(matches)}")
    return matches[0]

def one_input_dir(name: str) -> Path:
    matches = [path for path in INPUT_ROOT.rglob(name) if path.is_dir() and not path.is_symlink()]
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one input directory named {name}; found {len(matches)}")
    return matches[0]

symlinks = [str(path.relative_to(INPUT_ROOT)) for path in sorted(INPUT_ROOT.rglob("*")) if path.is_symlink()]
if symlinks:
    raise RuntimeError(f"Kaggle input contains forbidden symlinks: {symlinks[:3]}")
input_files = {
    str(path.relative_to(INPUT_ROOT)): {"sha256": sha256(path), "size": path.stat().st_size}
    for path in sorted(INPUT_ROOT.rglob("*"))
    if path.is_file() and not path.is_symlink()
}
input_inventory_sha256 = hashlib.sha256(
    json.dumps(input_files, separators=(",", ":"), sort_keys=True).encode()
).hexdigest()
runtime_pair = (sys.version_info[:2], np.__version__)
wheel = one_input_file(WHEEL_NAME)
observed_wheel_sha256 = sha256(wheel)
preflight = {
    "schema_version": "frem-s12-kaggle-preflight-v3",
    "scientific_admitted": False,
    "mode": MODE,
    "execute": EXECUTE,
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "numpy_version": np.__version__,
    "platform": platform.platform(),
    "input_exists": INPUT_ROOT.is_dir(),
    "input_files": input_files,
    "input_inventory_sha256": input_inventory_sha256,
    "wheel_name": WHEEL_NAME,
    "wheel_relative_path": str(wheel.relative_to(INPUT_ROOT)),
    "wheel_expected_sha256": WHEEL_SHA256,
    "wheel_observed_sha256": observed_wheel_sha256,
    "package_version": PACKAGE_VERSION,
}
write_new_json(OUTPUT / "preflight.json", preflight)
if SUPPORTED_RUNTIMES.get(runtime_pair[0]) != runtime_pair[1]:
    raise RuntimeError(f"Unsupported Kaggle runtime {platform.python_version()} / NumPy {np.__version__}; preflight receipt preserved")

if observed_wheel_sha256 != WHEEL_SHA256:
    raise RuntimeError(f"Wheel hash mismatch: {observed_wheel_sha256}; preflight receipt preserved")
thread_env = {
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "VECLIB_MAXIMUM_THREADS": "1",
    "NUMEXPR_NUM_THREADS": "1",
}
launcher_env = {
    **os.environ,
    **thread_env,
    "PYTHONHASHSEED": "0",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
    "PIP_NO_INPUT": "1",
    "PYTHONWARNINGS": "error",
    "PYTHONPATH": str(SITE),
}
install_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        "--no-compile",
        "--target",
        str(SITE),
        str(wheel),
]
installed = subprocess.run(install_command, capture_output=True, text=True, env=launcher_env)
verify_command = [
    sys.executable,
    "-c",
    "import json; import importlib.metadata as m; from frem_s12.evidence import package_identity; "
    "print(json.dumps({'version': m.version('frem-s12'), 'package': package_identity()}, sort_keys=True))",
]
verified = subprocess.run(verify_command, capture_output=True, text=True, env=launcher_env) if installed.returncode == 0 else None
installation = {
    "schema": "frem-s12-isolated-install-v1",
    "command": install_command,
    "returncode": installed.returncode,
    "stdout": installed.stdout,
    "stderr": installed.stderr,
    "target": str(SITE),
    "thread_environment": thread_env,
    "verification_returncode": None if verified is None else verified.returncode,
    "verification_stdout": None if verified is None else verified.stdout,
    "verification_stderr": None if verified is None else verified.stderr,
}
write_new_json(OUTPUT / "installation.json", installation)
if installed.returncode != 0:
    raise RuntimeError("Isolated wheel installation failed; installation.json preserved")
if verified is None or verified.returncode != 0:
    raise RuntimeError("Isolated package verification failed; installation.json preserved")
verified_value = json.loads(verified.stdout)
if verified_value.get("version") != PACKAGE_VERSION:
    raise RuntimeError(f"Installed package version mismatch: {verified_value.get('version')}")

In [ ]:
import os

base = [sys.executable, "-m", "frem_s12.cli"]
if MODE == "qualify-kaggle":
    command = [*base, "qualify-kaggle", "--output", str(OUTPUT / "environment.json")]
elif MODE == "generated-smoke":
    command = [*base, "generated-smoke", "--seed", str(SEED), "--output", str(OUTPUT / "generated-smoke")]
elif MODE == "run-search-cell":
    if not EXECUTE:
        raise RuntimeError("Scientific execution requires explicit EXECUTE=True")
    command = [
        *base,
        "run-search-cell",
        "--execute",
        "--trial",
        str(TRIAL),
        "--family",
        FAMILY,
        "--seed",
        str(SEED),
        "--archive",
        str(one_input_file(f"{FAMILY}-201608.zip")),
        "--acquisition-receipt",
        str(one_input_file(f"{FAMILY}-201608.zip.receipt.json")),
        "--data-receipt",
        str(one_input_file(f"{FAMILY}-data.json")),
        "--environment-receipt",
        str(one_input_file("environment.json")),
        "--consumption-root",
        str(OUTPUT / "consumption"),
        "--output",
        str(OUTPUT / f"trial={TRIAL}" / f"family={FAMILY}" / f"seed={SEED}"),
    ]
elif MODE == "select-search":
    command = [
        *base,
        "select-search",
        "--campaign",
        str(one_input_dir("complete-search")),
        "--output",
        str(OUTPUT / "selection.json"),
    ]
elif MODE == "run-cell":
    if not EXECUTE:
        raise RuntimeError("Scientific execution requires explicit EXECUTE=True")
    command = [
        *base,
        "run-cell",
        "--execute",
        "--family",
        FAMILY,
        "--fold",
        FOLD,
        "--seed",
        str(SEED),
        "--archive",
        str(one_input_file(f"{FAMILY}-201608.zip")),
        "--acquisition-receipt",
        str(one_input_file(f"{FAMILY}-201608.zip.receipt.json")),
        "--data-receipt",
        str(one_input_file(f"{FAMILY}-data.json")),
        "--environment-receipt",
        str(one_input_file("environment.json")),
        "--selection-receipt",
        str(one_input_file("selection.json")),
        "--consumption-root",
        str(OUTPUT / "consumption"),
        "--checkpoint-dir",
        str(OUTPUT / "checkpoints"),
        "--output",
        str(OUTPUT / f"family={FAMILY}" / f"fold={FOLD}" / f"seed={SEED}"),
    ]
elif MODE == "analyze":
    command = [
        *base,
        "analyze",
        "--campaign",
        str(one_input_dir("complete-campaign")),
        "--output",
        str(OUTPUT / "decision.json"),
    ]
else:
    raise RuntimeError(f"Unknown MODE: {MODE}")
print(" ".join(command))
completed = subprocess.run(command, capture_output=True, text=True, env=launcher_env)
print(completed.stdout)
print(completed.stderr, file=sys.stderr)
command_result = {
    "schema": "frem-s12-command-result-v1",
    "command": command,
    "returncode": completed.returncode,
    "stdout": completed.stdout,
    "stderr": completed.stderr,
}
write_new_json(OUTPUT / "command.json", command_result)
postcondition_error = None
if completed.returncode != 0:
    postcondition_error = f"FREM-S12 command failed with return code {completed.returncode}"
elif MODE == "qualify-kaggle":
    qualification = json.loads((OUTPUT / "environment.json").read_text())
    if qualification.get("scientific_admitted") is not True:
        postcondition_error = "Kaggle scientific qualification failed closed; preserve and inspect environment.json"

In [ ]:
import hashlib
import json

output_symlinks = [str(path.relative_to(OUTPUT)) for path in sorted(OUTPUT.rglob("*")) if path.is_symlink()]
output_special = [
    str(path.relative_to(OUTPUT))
    for path in sorted(OUTPUT.rglob("*"))
    if not path.is_symlink() and not path.is_file() and not path.is_dir()
]
if output_symlinks or output_special:
    postcondition_error = f"Kaggle output contains forbidden symlinks or special files: {(output_symlinks + output_special)[:3]}"
files = {
    str(path.relative_to(OUTPUT)): {"sha256": sha256(path), "size": path.stat().st_size}
    for path in sorted(OUTPUT.rglob("*"))
    if path.is_file() and path.name != "run-manifest.json" and not path.is_symlink()
}
manifest = {
    "schema": "frem-s12-kaggle-run-manifest-v1",
    "mode": MODE,
    "execute": EXECUTE,
    "package_version": PACKAGE_VERSION,
    "wheel_sha256": WHEEL_SHA256,
    "input_inventory_sha256": input_inventory_sha256,
    "thread_environment": thread_env,
    "command": command,
    "files": files,
}
write_new_json(OUTPUT / "run-manifest.json", manifest)
print(json.dumps(manifest, indent=2, sort_keys=True))
if postcondition_error is not None:
    raise RuntimeError(postcondition_error)